# Structured request adaptation: LoRA versus QLoRA

One workflow selects **CUDA → MPS → CPU**. With at least two CUDA GPUs, LoRA and QLoRA run concurrently on separate devices; otherwise they run sequentially. Hardware-derived batch sizes are frozen at run creation.

The notebook starts with a **development run: 128 training requests, 32 validation requests and one epoch**. Remove `preset="development"` in the configuration cell for the full five-epoch comparison. Install requirements, inspect the paired pilot, then continue. Each code cell performs one main action; imports and plots never start experiments. See `../README.md` for recovery and unattended execution.

Import standard-library notebook utilities.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

Set the project location. Edit this absolute path if you extracted the project elsewhere.

In [ ]:
PROJECT_ROOT = Path("/workspace/structured-request-adaptation")  # Edit for another location.
PROJECT_ROOT

Use the project directory as the working directory. This also recovers a kernel whose previous working directory was replaced.

In [ ]:
os.chdir(PROJECT_ROOT)

Install requirements into the current notebook Python with pip. This cell installs packages only; it does not download models or run experiments. If pip updates packages already imported by this kernel, restart the kernel, rerun the imports/path cells above, then continue below. You can skip this install cell once installation has succeeded.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])

Make the project source modules available to the notebook.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))

Import the controller and typed configuration. These imports do not start experiments.

In [ ]:
from src.launcher import ExperimentLauncher
from src.config import ExperimentConfig

Create the lightweight experiment controller.

In [ ]:
launcher = ExperimentLauncher(PROJECT_ROOT)

Detect available devices in a worker using the current notebook Python. CUDA takes priority, then MPS, then CPU. Inspect the selected device(s), precision, execution mode and batch defaults.

In [ ]:
hardware = launcher.inspect_hardware()
hardware

Start with the development preset: 128 training examples, 32 validation examples and one epoch. This still downloads the complete model. Remove `preset="development"` below for the full five-epoch comparison on all official examples. MPS, CPU and smaller CUDA devices use microbatch 1 and accumulation 16; larger CUDA devices use 4 and 4. Both arms share an effective batch of 16.

In [ ]:
config = ExperimentConfig.for_hardware(
    hardware,
    model_revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    preset="development",
)
config

Choose a fresh run or explicitly name an existing compatible run. Creating a run again with None intentionally allocates the next run number.

In [ ]:
RUN_TO_RESUME = None  # Example: "run_001"

Create the run and record its hardware plan, or verify the explicitly named run and register a compatible replacement session. A change of device type, batching, precision, source or dependencies requires a fresh run.

In [ ]:
run_record = launcher.run_action("create_run", config=config.to_dict()) if RUN_TO_RESUME is None else launcher.run_action("resume_session", run_name=RUN_TO_RESUME)
run_record

Reuse this exact run name throughout the notebook.

In [ ]:
RUN_NAME = run_record["run_name"]
RUN_NAME

Acquire the official MASSIVE archive once into the project-level `data/` folder and verify the shared cache. Later runs reuse its verified archive, English JSONL and license. Missing or changed committed files stop acquisition instead of silently changing dataset identity.

In [ ]:
dataset = launcher.run_action("acquire_dataset", run_name=RUN_NAME)
dataset

### Prepare structured targets

Parse annotations, retain repeated slots, preserve official partitions, and audit duplicate requests. Gold annotations are never included in request prompts.


In [ ]:
prepared = launcher.run_action("prepare_data", run_name=RUN_NAME)
prepared

### Inspect the data audit

Read the saved audit without starting another experiment.


In [ ]:
data_audit = json.loads((PROJECT_ROOT / "runs" / RUN_NAME / "preparation" / "data.json").read_text())
data_audit

### Acquire the pinned model and tokenizer

Download the immutable Qwen revision into this run. This stage can take time on first execution.


In [ ]:
model_inputs = launcher.run_action("acquire_model", run_name=RUN_NAME)
model_inputs

### Prepare NF4 on its assigned device

Create the saved double-quantized NF4 base using the QLoRA GPU. Later QLoRA evaluation reloads this snapshot and the unmerged adapter.


In [ ]:
quantized_inputs = launcher.run_action("prepare_quantized_base", run_name=RUN_NAME)
quantized_inputs

### Build prompts and tokenize

Tokenize each official partition with the fixed instruction, enforce sequence limits, and mask all prompt tokens from the training loss.


In [ ]:
tokenization = launcher.run_action("tokenize_data", run_name=RUN_NAME)
tokenization

### Inspect answer-only masking

Show a prepared example and its supervised token boundaries before numerical checks.


In [ ]:
masking_example = launcher.run_action("inspect_example", run_name=RUN_NAME)
masking_example

### Inspect the architecture

Save model configuration, tensor shapes and adapter targets. Header inspection does not run training.


In [ ]:
architecture = launcher.run_action("inspect_architecture", run_name=RUN_NAME)
architecture

### Check LoRA on its assigned device

Create the single shared initial adapter and verify a synthetic forward/backward/update and adapter reconstruction. This is a compatibility check, not an experimental training epoch.


In [ ]:
lora_preflight = launcher.run_action("preflight_lora", run_name=RUN_NAME)
lora_preflight

### Check QLoRA on its assigned device

Load the same initial adapter into the saved NF4 model and verify numerical compatibility and reconstruction on its assigned GPU.


In [ ]:
qlora_preflight = launcher.run_action("preflight_qlora", run_name=RUN_NAME)
qlora_preflight

### Commit the paired compatibility gate

Both device-specific preflights must pass with matching trainable parameter inventories before baseline or training jobs can start.


In [ ]:
paired_preflight = launcher.run_action("preflight", run_name=RUN_NAME)
paired_preflight

Start the LoRA pilot on its assigned device. Its updated adapter is discarded; main training checkpoints are untouched.

In [ ]:
lora_pilot = launcher.start_arm(RUN_NAME, "lora", phase="pilot")

Schedule the QLoRA pilot. It starts concurrently when two CUDA devices are assigned, or is queued behind LoRA on a single device.

In [ ]:
qlora_pilot = launcher.start_arm(RUN_NAME, "qlora", phase="pilot")

Wait for both pilots and inspect their activity, phase timings and memory. These measurements are workload samples, not a promise of total completion time. Continue below only after reviewing them against your budget.

In [ ]:
pilot_results = launcher.wait([lora_pilot, qlora_pilot])
pilot_results

Inspect the compact pilot summary. Training projections exclude loading, checkpoint writes, validation, test and benchmarks. Parallel training follows the slower arm; serial execution adds their times. Activity errors mean unavailable measurements, not zero utilization.

In [ ]:
pilot_summary = {
    arm: {
        "measured_updates": record["result"]["measured_updates"],
        "median_seconds_per_update": record["result"]["training_projection"]["seconds_per_update_median"],
        "projected_training_only_hours": record["result"]["training_projection"]["training_compute_seconds_at_median"] / 3600,
        "validation_sample_examples": record["result"]["validation_sample"]["examples"],
        "validation_sample_generation_seconds": record["result"]["validation_sample"]["generation_seconds"],
        "maximum_observed_memory_bytes": record["result"]["activity"]["maximum_observed_memory_bytes"],
        "activity_sampling_errors": record["result"]["activity"]["sampling_errors"],
    }
    for arm, record in pilot_results.items()
}
pilot_summary

Schedule the LoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
lora_baseline = launcher.start_arm(RUN_NAME, "lora", phase="baseline")

Schedule the QLoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
qlora_baseline = launcher.start_arm(RUN_NAME, "qlora", phase="baseline")

### Wait for both baseline validation jobs

Display progress and saved log locations while both workers run. Any worker failure stops its peer and raises an error. Interrupting this wait stops both processes; re-run the phase to recover from the last committed checkpoint.


In [ ]:
baseline_results = launcher.wait([lora_baseline, qlora_baseline])
baseline_results

Schedule the LoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
lora_training = launcher.start_arm(RUN_NAME, "lora", phase="training")

Schedule the QLoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
qlora_training = launcher.start_arm(RUN_NAME, "qlora", phase="training")

### Wait for both training jobs

Display progress and saved log locations while both workers run. Any worker failure stops its peer and raises an error. Interrupting this wait stops both processes; re-run the phase to recover from the last committed checkpoint.


In [ ]:
training_results = launcher.wait([lora_training, qlora_training])
training_results

### Inspect LORA saved history

Read saved epoch metrics only; this does not launch training.


In [ ]:
lora_history = [json.loads(path.read_text()) for path in sorted((PROJECT_ROOT / "runs" / RUN_NAME / "trials" / "lora" / "metrics").glob("epoch_??.json"))]
lora_history

### Inspect QLORA saved history

Read saved epoch metrics only; this does not launch training.


In [ ]:
qlora_history = [json.loads(path.read_text()) for path in sorted((PROJECT_ROOT / "runs" / RUN_NAME / "trials" / "qlora" / "metrics").glob("epoch_??.json"))]
qlora_history

### Freeze the final comparison

This cell runs only after both training processes have finished. It verifies matching update/token budgets and locks final checkpoint identities and generation settings before test access.


In [ ]:
frozen = launcher.run_action("freeze_evaluation", run_name=RUN_NAME)
frozen

Schedule the LoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
lora_test = launcher.start_arm(RUN_NAME, "lora", phase="test")

Schedule the QLoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
qlora_test = launcher.start_arm(RUN_NAME, "qlora", phase="test")

### Wait for both test evaluation jobs

Display progress and saved log locations while both workers run. Any worker failure stops its peer and raises an error. Interrupting this wait stops both processes; re-run the phase to recover from the last committed checkpoint.


In [ ]:
test_results = launcher.wait([lora_test, qlora_test])
test_results

Schedule the LoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
lora_benchmark = launcher.start_arm(RUN_NAME, "lora", phase="benchmark")

Schedule the QLoRA action. With two CUDA devices it can run alongside the other arm; on one device the controller queues the second arm until the first finishes.

In [ ]:
qlora_benchmark = launcher.start_arm(RUN_NAME, "qlora", phase="benchmark")

### Wait for both inference benchmarks jobs

Display progress and saved log locations while both workers run. Any worker failure stops its peer and raises an error. Interrupting this wait stops both processes; re-run the phase to recover from the last committed checkpoint.


In [ ]:
benchmark_results = launcher.wait([lora_benchmark, qlora_benchmark])
benchmark_results

### Collect the comparison

Read verified saved results. Missing or failed stages remain explicit; collection never fills gaps by launching experiments.


In [ ]:
comparison = launcher.run_action("collect_results", run_name=RUN_NAME)
comparison

### Generate training and validation loss

Generate one figure from saved measurements. Concurrent execution and one-seed limitations apply.


In [ ]:
plot_losses_result = launcher.run_action("plot_losses", run_name=RUN_NAME)
plot_losses_result

### Generate validation correctness and schema validity

Generate one figure from saved measurements. Concurrent execution and one-seed limitations apply.


In [ ]:
plot_validation_result = launcher.run_action("plot_validation", run_name=RUN_NAME)
plot_validation_result

### Generate four-system task metrics

Generate one figure from saved measurements. Concurrent execution and one-seed limitations apply.


In [ ]:
plot_task_metrics_result = launcher.run_action("plot_task_metrics", run_name=RUN_NAME)
plot_task_metrics_result

### Generate observed memory and runtime

Generate one figure from saved measurements. Concurrent execution and one-seed limitations apply.


In [ ]:
plot_memory_runtime_result = launcher.run_action("plot_memory_runtime", run_name=RUN_NAME)
plot_memory_runtime_result

### Generate prediction disagreements and failures

Generate one figure from saved measurements. Concurrent execution and one-seed limitations apply.


In [ ]:
plot_disagreements_result = launcher.run_action("plot_disagreements", run_name=RUN_NAME)
plot_disagreements_result

### Write the technical report

Generate the report from saved results and include the the selected device execution protocol and hardware provenance. Incomplete evidence never becomes a claimed result.


In [ ]:
report = launcher.run_action("write_report", run_name=RUN_NAME)
report

List saved figures without regenerating results.

In [ ]:
figure_paths = sorted((PROJECT_ROOT / "runs" / RUN_NAME / "figures").glob("*.png"))
figure_paths

### Choose a prediction request

Optional: edit this request after completing the comparison. Prediction does not change adapters or evaluation settings.


In [ ]:
PREDICTION_REQUEST = "set an alarm for seven am"

### Reload QLoRA on its assigned device and predict

Reload the saved NF4 base and final unmerged QLoRA adapter on its assigned device, predict once, then release the model. The response is not repaired.


In [ ]:
prediction = launcher.run_action("predict", run_name=RUN_NAME, arm="qlora", text=PREDICTION_REQUEST)
prediction

### Show run status and artifact locations

Read the final status, including missing measurements. Preserve the shared data and run artifacts when saving the selected device outputs.


In [ ]:
status = launcher.run_action("run_summary", run_name=RUN_NAME)
status